In [1]:
instruction_path = "/qumulo/shared_data/aofei_summer/data/RegAlign/final_data/Instruct_71k_1104.json"
import json
with open(instruction_path, "r") as f:
    data = json.load(f)

In [2]:
data[0]

{'image': ['images_all/pmc_81861_0.jpg'],
 'conversations': [{'from': 'human',
   'value': 'What is the subcellular localization of the ABHD17A protein in COS-7 cells based on this image?'},
  {'from': 'gpt',
   'value': 'According to the image, the ABHD17A protein is localized to the plasma membrane and various endosomal compartments, including early endosomes (Rab5), late endosomes (Rab7), and recycling endosomes (Rab11). The protein is also found in the Golgi apparatus, as indicated by the GM130 marker. The image also shows the localization of a mutant form of ABHD17A (ABHD17A ΔN) in relation to the Golgi apparatus. Additionally, the image demonstrates the co-expression of wild-type and mutant forms of mCherry-tagged ABHD17A with EGFP-N-Ras in COS-7 cells.'}],
 'id': 'Instruction-Tuning_81861',
 'modality': 'Microscopy Images',
 'body_part': 'Cell',
 'mask_files': None,
 'mask_codes': None,
 'mask_orders': []}

In [3]:
import json
import re
import shutil
from pathlib import Path

src = Path("/qumulo/shared_data/aofei_summer/data/RegAlign/final_data/Instruct_71k_1104.json")
backup = src.with_suffix(src.suffix + ".bak")
out = src.with_name(src.stem + "_seg_replaced" + src.suffix)

# backup
shutil.copy2(src, backup)

pattern = re.compile(r"\[M\d+_\d+\]")

with open(src, "r", encoding="utf-8") as f:
    data = json.load(f)

total_replacements = 0
items_changed = 0

for item in data:
    convs = item.get("conversations", [])
    changed = False
    for conv in convs:
        if isinstance(conv, dict):
            val = conv.get("value")
            if isinstance(val, str):
                new_val, n = pattern.subn("[SEG]", val)
                if n:
                    conv["value"] = new_val
                    total_replacements += n
                    changed = True
    if changed:
        items_changed += 1

with open(out, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Items changed: {items_changed}, total tokens replaced: {total_replacements}")
print(f"Backup saved to: {backup}")
print(f"Output written to: {out}")

Items changed: 12337, total tokens replaced: 27079
Backup saved to: /qumulo/shared_data/aofei_summer/data/RegAlign/final_data/Instruct_71k_1104.json.bak
Output written to: /qumulo/shared_data/aofei_summer/data/RegAlign/final_data/Instruct_71k_1104_seg_replaced.json


In [6]:
data[-3]

{'id': 12505,
 'image': '/qumulo/shared_data/aofei_summer/data/BiomedParse/UWaterlooSkinCancer/UWaterlooSkinCancer/train/dermIS_SSM16_dermoscopy_skin.png',
 'conversations': [{'from': 'human',
   'value': "We'll examine this dermoscopy image. Be prepared to segment lesions and note any malignancy."},
  {'from': 'assistant',
   'value': 'Understood — I can segment the lesion and indicate if features are consistent with melanoma.'},
  {'from': 'human', 'value': 'Segment the visible lesion first.'},
  {'from': 'assistant', 'value': 'The lesion is segmented as [SEG].'},
  {'from': 'human',
   'value': 'Is there evidence of melanoma? If yes, provide the diagnosis and segmentation.'},
  {'from': 'assistant',
   'value': 'Findings consistent with melanoma are present and segmented as [SEG].'}],
 'mask_files': {'28': '/qumulo/shared_data/aofei_summer/data/BiomedParse/UWaterlooSkinCancer/UWaterlooSkinCancer/train_mask/dermIS_SSM16_dermoscopy_skin_melanoma.png',
  '29': '/qumulo/shared_data/aofe